# S6_2 — Embedding Space Analysis

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Produce the t-SNE / UMAP progression figure — the single strongest visual argument for RQ1.
Shows representation quality improving across: raw pixels → generic I-JEPA → Leaf-JEPA → PEFT.

**Outputs:**
- `tsne_progression_3panel.png` — flagship RQ1 figure
- `umap_progression_3panel.png` — robustness check
- `silhouette_progression.json` — quantitative progression
- `cluster_distances.json` — intra/inter-class distances
- `hard_pair_separation.csv` — centroid distances for confusing pairs
- `tsne_hard_pairs_focused.png` — focused plot of most-confused classes

**Research Questions Served:** RQ1 (primary), RQ4 (cross-domain representations)


## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import OrderedDict
import torch

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, get_device, load_json, save_json,
    load_ijepa_encoder, SimpleImageDataset, get_eval_transform,
    load_split_data, extract_features,
    compute_tsne, compute_umap, compute_silhouette,
    compute_cluster_distances, hard_pair_separation,
    plot_tsne_grid, print_section, HAS_UMAP,
)
from torch.utils.data import DataLoader

set_seed(RANDOM_SEED)
device = get_device()
print(f"Device: {device}")

for d in [STAGE6_FIGURES, STAGE6_DATA]:
    d.mkdir(parents=True, exist_ok=True)


Device: cuda


## Load Test Data

In [ ]:
print_section("Loading Test Data")

image_paths, labels, class_names = load_split_data(SPLITS_DIR, split="test")
labels = np.array(labels)
print(f"  Test samples: {len(image_paths)}")
print(f"  Classes: {len(class_names)}")

# Subsample for t-SNE speed if needed
if len(image_paths) > MAX_SAMPLES_EMBED:
    rng = np.random.RandomState(RANDOM_SEED)
    idx = rng.choice(len(image_paths), MAX_SAMPLES_EMBED, replace=False)
    image_paths = [image_paths[i] for i in idx]
    labels = labels[idx]
    print(f"  Subsampled to {MAX_SAMPLES_EMBED} for t-SNE")

transform = get_eval_transform(NORM_MEAN, NORM_STD, IMAGE_SIZE)
dataset = SimpleImageDataset(image_paths, labels, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


## Extract features

In [ ]:
# Extract features from all encoders
print_section("Feature Extraction")

all_features = OrderedDict()

# 1. Raw pixel features (flatten and PCA — matching Stage 2)
print("  Extracting raw pixel features...")
from torchvision import transforms as T
raw_transform = T.Compose([T.Resize(64), T.CenterCrop(64), T.ToTensor()])
raw_ds = SimpleImageDataset(image_paths, labels, transform=raw_transform)
raw_loader = DataLoader(raw_ds, batch_size=64, shuffle=False, num_workers=2)
raw_feats = []
raw_labels = []
for imgs, lbls in raw_loader:
    raw_feats.append(imgs.flatten(1).numpy())
    raw_labels.append(lbls.numpy() if isinstance(lbls, torch.Tensor) else np.array(lbls))
raw_feats = np.concatenate(raw_feats)
raw_labels_arr = np.concatenate(raw_labels)
all_features["Raw Pixels"] = raw_feats
print(f"    Shape: {raw_feats.shape}")

# 2. Generic I-JEPA
if IJEPA_CHECKPOINT.exists():
    print("  Loading generic I-JEPA encoder...")
    generic_model = load_ijepa_encoder(IJEPA_CHECKPOINT, device=device)
    generic_feats, _ = extract_features(generic_model, dataloader, device=device)
    all_features["Generic I-JEPA"] = generic_feats
    print(f"    Shape: {generic_feats.shape}")
    del generic_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("  \u26a0\ufe0f Generic I-JEPA checkpoint not found — skipping")

# 3. Leaf-JEPA
if LEAF_JEPA_CHECKPOINT.exists():
    print("  Loading Leaf-JEPA encoder...")
    leaf_model = load_ijepa_encoder(LEAF_JEPA_CHECKPOINT, device=device)
    leaf_feats, _ = extract_features(leaf_model, dataloader, device=device)
    all_features["Leaf-JEPA"] = leaf_feats
    print(f"    Shape: {leaf_feats.shape}")
    del leaf_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("  \u26a0\ufe0f Leaf-JEPA checkpoint not found — skipping")

print(f"\n  Feature sets available: {list(all_features.keys())}")


## Silhouette Score

In [ ]:
# Compute silhouette scores
print_section("Silhouette Scores")

silhouette_scores = {}
for name, feats in all_features.items():
    pca_dim = PCA_COMPONENTS if feats.shape[1] > PCA_COMPONENTS else None
    score = compute_silhouette(feats, labels, pca_dim=pca_dim, seed=RANDOM_SEED)
    silhouette_scores[name] = score
    print(f"  {name}: {score:.4f}")

save_json(silhouette_scores, STAGE6_DATA / "silhouette_progression.json")

# Report progression
if len(silhouette_scores) >= 2:
    keys = list(silhouette_scores.keys())
    for i in range(1, len(keys)):
        delta = silhouette_scores[keys[i]] - silhouette_scores[keys[i-1]]
        print(f"  {keys[i-1]} -> {keys[i]}: {delta:+.4f}")


## t-SNE Progression

In [ ]:
print_section("t-SNE Progression")

tsne_coords = OrderedDict()
for name, feats in all_features.items():
    print(f"  Computing t-SNE for {name}...")
    pca_dim = PCA_COMPONENTS if feats.shape[1] > PCA_COMPONENTS else None
    coords = compute_tsne(feats, perplexity=TSNE_PERPLEXITY, n_iter=TSNE_N_ITER,
                          seed=TSNE_RANDOM_STATE, pca_dim=pca_dim)
    tsne_coords[name] = coords

plot_tsne_grid(tsne_coords, labels, class_names,
               STAGE6_FIGURES / "tsne_progression_3panel.png",
               silhouette_scores=silhouette_scores)


## UMAP Progresssion


In [ ]:
if HAS_UMAP:
    print_section("UMAP Progression")
    umap_coords = OrderedDict()
    for name, feats in all_features.items():
        print(f"  Computing UMAP for {name}...")
        pca_dim = PCA_COMPONENTS if feats.shape[1] > PCA_COMPONENTS else None
        coords = compute_umap(feats, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                              seed=RANDOM_SEED, pca_dim=pca_dim)
        if coords is not None:
            umap_coords[name] = coords

    if umap_coords:
        plot_tsne_grid(umap_coords, labels, class_names,
                       STAGE6_FIGURES / "umap_progression_3panel.png",
                       silhouette_scores=silhouette_scores)
else:
    print("\u26a0\ufe0f UMAP not installed — run: pip install umap-learn")


## Cluster Distance and Hard-pair sepration

In [ ]:
print_section("Cluster Distance Analysis")

cluster_data = {}
for name, feats in all_features.items():
    if feats.shape[1] > PCA_COMPONENTS:
        from sklearn.decomposition import PCA as PCA_
        pca = PCA_(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED)
        feats_r = pca.fit_transform(feats)
    else:
        feats_r = feats
    cd = compute_cluster_distances(feats_r, labels, class_names)
    cluster_data[name] = cd
    avg_intra = np.mean(list(cd["intra_class"].values()))
    avg_inter = np.mean(list(cd["inter_class"].values()))
    print(f"  {name}: avg intra={avg_intra:.4f}, avg inter={avg_inter:.4f}, ratio={avg_inter/avg_intra:.2f}")

save_json(cluster_data, STAGE6_DATA / "cluster_distances.json")

# Hard-pair separation
# Load Stage 2 top confused pairs if available
top_pairs_path = ANALYSIS_DIR / "top_confused_pairs.json" if ANALYSIS_DIR else None
hard_pairs = []
if top_pairs_path and top_pairs_path.exists():
    tp_data = load_json(top_pairs_path)
    for p in tp_data[:5]:
        # Find class indices
        try:
            idx_a = class_names.index(p.get("class_a", ""))
            idx_b = class_names.index(p.get("class_b", ""))
            hard_pairs.append((idx_a, idx_b))
        except ValueError:
            pass

if hard_pairs:
    print("\n  Hard-pair separation analysis:")
    sep_results = {}
    for name, feats in all_features.items():
        if feats.shape[1] > PCA_COMPONENTS:
            feats_r = PCA_(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED).fit_transform(feats)
        else:
            feats_r = feats
        seps = hard_pair_separation(feats_r, labels, hard_pairs, class_names)
        sep_results[name] = seps
        for s in seps:
            print(f"    {name}: {s['class_a']} vs {s['class_b']}: "
                  f"dist={s['centroid_distance']:.4f}, sep_ratio={s['separation_ratio']:.3f}")

    # Save as CSV
    rows = []
    for model_name, seps in sep_results.items():
        for s in seps:
            rows.append({"model": model_name, **s})
    pd.DataFrame(rows).to_csv(STAGE6_TABLES / "hard_pair_separation.csv", index=False)
else:
    print("  \u26a0\ufe0f No hard pairs found from Stage 2 — skipping focused analysis")
    print("  You can manually define pairs: hard_pairs = [(idx_a, idx_b), ...]")


## Focused t-SNE for hard pairs

In [ ]:
if hard_pairs and "Leaf-JEPA" in all_features:
    print_section("Hard-Pair Focused t-SNE")

    # Get just the hard pair classes
    pair_classes = set()
    for a, b in hard_pairs[:3]:  # Top 3 pairs
        pair_classes.add(a)
        pair_classes.add(b)
    pair_classes = sorted(pair_classes)

    mask = np.isin(labels, pair_classes)
    sub_labels = labels[mask]
    sub_names = [class_names[c] for c in pair_classes]

    # Remap labels for cleaner plotting
    label_map = {c: i for i, c in enumerate(pair_classes)}
    sub_labels_mapped = np.array([label_map[l] for l in sub_labels])

    focused_coords = OrderedDict()
    for name in ["Generic I-JEPA", "Leaf-JEPA"]:
        if name in all_features:
            sub_feats = all_features[name][mask]
            pca_dim = min(PCA_COMPONENTS, sub_feats.shape[1])
            coords = compute_tsne(sub_feats, perplexity=min(TSNE_PERPLEXITY, len(sub_feats)//4),
                                  seed=RANDOM_SEED, pca_dim=pca_dim)
            focused_coords[name] = coords

    if focused_coords:
        plot_tsne_grid(focused_coords, sub_labels_mapped, sub_names,
                       STAGE6_FIGURES / "tsne_hard_pairs_focused.png",
                       figsize_per_panel=(7, 6))

print("\n\u2705 S6_2 complete.")


## Critical Analysis

### Key Findings to Report
1. **Silhouette progression**: The increase from raw pixels to Leaf-JEPA quantifies representation improvement.
   A positive trend confirms RQ1.
2. **Hard-pair separation**: If the top confused pairs from Stage 2 have higher separation ratios
   after domain pretraining, this is direct evidence that Leaf-JEPA resolves the specific ambiguities
   that Stage 2 predicted would be problematic.
3. **t-SNE is qualitative, silhouette is quantitative**: Always cite the silhouette score alongside
   the t-SNE figure. The visual is compelling but subjective; the number is defensible.

### Identical Hyperparameters Requirement
All panels in the t-SNE figure use the same perplexity, number of iterations, and random state.
This is critical for a fair comparison — if you change any t-SNE hyperparameter between panels,
the comparison is invalid.
